# VaultBox Colab Server

Run the cells in order. When the connection box appears, copy **Server URL** and **Connection Token** into VaultBox -> Settings -> Connect to Colab.

Colab is only a temporary worker. Provider accounts stay inside VaultBox.

In [ ]:
#@title 1. Install dependencies { display-mode: "form" }
import subprocess, sys
def quiet(cmd):
    subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
quiet(['apt-get', 'update', '-qq'])
quiet(['apt-get', 'install', '-y', '-qq', 'p7zip-full', 'unrar', 'git', 'curl'])
quiet([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
print('Dependencies ready')


In [ ]:
#@title 2. Download latest VaultBox Colab source { display-mode: "form" }
import os, shutil, subprocess, pathlib, sys
REPO = os.environ.get('VAULTBOX_PUBLIC_REPO', 'https://github.com/CeatursHarmginton/vault-box.git')
ROOT = pathlib.Path('/content/vaultbox-public')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.check_call(['git', 'clone', '--depth', '1', REPO, str(ROOT)], stdout=subprocess.DEVNULL)
os.chdir(ROOT / 'colab')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], stdout=subprocess.DEVNULL)
print('VaultBox Colab source ready')


In [ ]:
#@title 3. Start server { display-mode: "form" }
import os, secrets, subprocess, sys, time, html
from IPython.display import HTML, display
from google.colab import output
os.environ['COLAB_SERVER_TOKEN'] = os.environ.get('COLAB_SERVER_TOKEN') or secrets.token_urlsafe(32)
server_url = output.eval_js('google.colab.kernel.proxyPort(8000)')
os.environ['COLAB_PUBLIC_URL'] = server_url
proc = subprocess.Popen([sys.executable, 'scripts/start_colab_server.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(3)
token = os.environ['COLAB_SERVER_TOKEN']
display(HTML(f'''
<div style="font-family:system-ui,-apple-system,Segoe UI,sans-serif;border:1px solid #d0d7de;border-radius:10px;padding:16px;max-width:760px;background:#fff">
  <h2 style="margin:0 0 10px;font-size:20px">VaultBox Colab Server is ready</h2>
  <p style="margin:0 0 14px;color:#57606a">Copy these into VaultBox -> Settings -> Connect to Colab.</p>
  <label style="display:block;font-weight:600;margin:10px 0 6px">Server URL</label>
  <div style="display:flex;gap:8px;align-items:center">
    <input id="vaultbox-server-url" readonly value="{html.escape(server_url)}" style="flex:1;padding:10px;border:1px solid #d0d7de;border-radius:6px;font:13px ui-monospace,SFMono-Regular,Consolas,monospace">
    <button onclick="navigator.clipboard.writeText(document.getElementById('vaultbox-server-url').value);this.textContent='Copied'" style="padding:10px 12px;border:1px solid #d0d7de;border-radius:6px;background:#f6f8fa;cursor:pointer">Copy URL</button>
  </div>
  <label style="display:block;font-weight:600;margin:14px 0 6px">Connection Token</label>
  <div style="display:flex;gap:8px;align-items:center">
    <input id="vaultbox-token" readonly value="{html.escape(token)}" style="flex:1;padding:10px;border:1px solid #d0d7de;border-radius:6px;font:13px ui-monospace,SFMono-Regular,Consolas,monospace">
    <button onclick="navigator.clipboard.writeText(document.getElementById('vaultbox-token').value);this.textContent='Copied'" style="padding:10px 12px;border:1px solid #d0d7de;border-radius:6px;background:#f6f8fa;cursor:pointer">Copy Token</button>
  </div>
</div>
'''))


In [ ]:
#@title 4. Keep server alive { display-mode: "form" }
import time
print('Worker running. Keep this cell active while transferring.')
while True:
    line = proc.stdout.readline() if proc.stdout else ''
    if line and ('ERROR' in line.upper() or 'TRACEBACK' in line.upper()):
        print(line.rstrip())
    time.sleep(1)
